In [ ]:
from main import*
from run_estimator import*
from datasets import ring

from IPython.display import clear_output # To clear tqdm bars

Generate half-ring point clouds for different number of samples. Each dataset is modified by moving some points to create a shortcut in the middle with approx. $\sqrt n$ points.

In [ ]:
base = 2
start = 7
end = 12
k = end - start + 1
n_samples = np.logspace(start, end, num=k, base=base, dtype=int) # Number of samples range from base**start to base**stop
K = len(n_samples)

print(f'Number of samples {n_samples}')

precision_segment_start = 6 # Precision for identifying geodesics
precision_segment_final = 500 # Precision for the final computation of FDTM

avg = 200 # Number of repetitions for averaging stats

In [ ]:
r = 2 # Radius

seed = 17 # Random seed

points, points_shortcut = [], [] # Datasets
X_init = [[-2, 0], [2, 0]]  # Endpoints to be forced into the dataset
for n in n_samples:
    points.append([ring(n, r, X_init=X_init, shortcut=False, only_top=True, seed=seed+a) for a in range(avg)])
    points_shortcut.append([ring(n, r, X_init=X_init, shortcut=True, only_top=True, seed=seed+a) for a in range(avg)])

In [ ]:
color_points = 'gray'

fig, axes = plt.subplots(1, 2, figsize=(10, 20))
for ax, X in zip(axes, (points[-1][0], points_shortcut[-1][0])):
    ax.scatter(X[:,0], X[:,1], c=color_points, s=20, marker='x')
    ax.axis('off')
    ax.set_aspect('equal', adjustable='box')

fig.suptitle(f'Example of dataset with and without shortcut, n={n_samples[-1]}')
fig.subplots_adjust(top=1.7)
plt.show()

Experiments

In [ ]:
experiment, experiment_shortcut = {}, {}  # To store all the data

Fermat

In [ ]:
alpha = 1.25  # Sample Fermat parameter
knns = np.array(np.maximum(1, np.log(n_samples)), dtype=int)  # Number of nearest neighbours
show_admissible = False

name = 'Fermat'

experiment[name] = run_estimator(Fermat, points, knns=knns, alpha=alpha, avg=avg)
experiment_shortcut[name] = run_estimator(Fermat, points_shortcut, knns=knns, alpha=alpha, avg=avg)

experiment[name].color = 'orange'
experiment[name].linestyle = '-'
experiment_shortcut[name].color = 'orange'
experiment_shortcut[name].linestyle = '-'

clear_output()

FDTM : We use an approximation of the empirical DTM by using only $\sqrt{n}$ points, to make computations faster. We also use a k-NN graph with $k = \sqrt{n}$.

In [ ]:
m = 0.05
p = 2
q = 0.25
dtm_arg = DTM_arg(m, p, q)

approx_power = 0.5

dtms = [[DTM(X[:int(len(X)**approx_power)+1], dtm_arg) for X in Xs] for Xs in points]
dtms_shortcut = [[DTM(X[:int(len(X)**approx_power)+1], dtm_arg) for X in Xs] for Xs in points_shortcut]

precision = 5  # Number of points to approximate the integral over edges
precisions=np.full(len(points), precision)

In [ ]:
knn_power = 1/2
knns = np.array(n_samples**knn_power, dtype=int)  # Number of nearest neighbours

name = 'FDTM'

experiment[name] = run_estimator(FDTM, points, dtms=dtms, knns=knns, precisions=precisions, avg=avg)
experiment_shortcut[name] = run_estimator(FDTM, points_shortcut, dtms=dtms_shortcut, knns=knns, precisions=precisions, avg=avg)

experiment[name].color = 'blue'
experiment[name].linestyle = '--'
experiment_shortcut[name].color = 'blue'
experiment_shortcut[name].linestyle = '--'

clear_output()

Plots

In [ ]:
save = True  # Save figures

In [ ]:
fig, ax = plt.subplots()
for name in experiment.keys():
    deviation = []
    ds = experiment[name].distances
    ds_shortcut = experiment_shortcut[name].distances
    for d0, d1 in zip(ds, ds_shortcut):
        filter = np.bitwise_and(d0<np.inf, d1<np.inf) # Ignore cases where some there is no path
        mean0, mean1 = np.average(d0[filter]), np.average(d1[filter])
        deviation.append(abs(mean0-mean1) / min(mean0, mean1)) # Ignore warning when some value is inf
    ax.plot(n_samples, deviation, label=name, c=experiment[name].color, linestyle=experiment[name].linestyle)

ax.legend(loc='lower left')
ax.set_xlabel('Number of sample points')
ax.set_ylabel('Relative offset')
ax.set_yscale('log')

ax.grid(True, linestyle='--', alpha=0.6)
ax.grid(True, which='minor', linestyle=':', alpha=0.3)

if save : fig.savefig(f"figures\\ring_offset.png", dpi=300)

ax.set_title('Deviation due to addition of line.')

plt.gcf().set_dpi(100)
plt.show()

In [ ]:
ncols = 2
nrows = K  # Number of different sample sizes
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows/2))
axes = np.atleast_2d(axes)

for l in range(2):
    P, E = (points, points_shortcut)[l], (experiment, experiment_shortcut)[l]
    for k, X in enumerate(P):
        axes[k][l].scatter(X[0][:,0], X[0][:,1], c=color_points, s=20, marker='x')
        for name, data in E.items():
            plot_kwargs = {'label' : f'{name} : {data.distances[k][0]:.4f} in {data.durations[k]:.2f}s',
                        'c' : data.color,
                        'linestyle' : data.linestyle}
            if data.paths[k] is not None : axes[k][l].plot(X[0][data.paths[k], 0], X[0][data.paths[k], 1], **plot_kwargs)
            else : axes[k][l].plot([], [], **plot_kwargs)  # If no path, plot nothing but display label

        axes[k][l].axis('off')
        axes[k][l].set_aspect('equal', adjustable='box')

        if save:  # Save plot before adding legend
            extent = axes[k][l].get_window_extent().transformed(fig.dpi_scale_trans.inverted())
            fig.savefig(f"figures\\ring\\n{len(X[0])}-{l}.png", bbox_inches=extent, dpi=300)

        axes[k][l].set_title(f'{len(X[0])} sample points')
        axes[k][l].legend(loc='upper left') 
    
plt.show()